In [4]:
import pybigtools
print(pybigtools.__version__)

0.3.0


In [5]:
import os
import glob

import numpy as np
import pandas as pd

### Data Retrieval

The following function retrieves the file path and name of all files in the /data directory.

In [7]:
DATA_DIR = "../../data"   # notebook is in models/linear_model/
WORK_DIR = "../.."        # so submission.csv lands in the project root

ALL_FILES = sorted(
    p for p in glob.glob(os.path.join(DATA_DIR, "**", "*"), recursive=True)
    if os.path.isfile(p)
)
print(f"{len(ALL_FILES)} files found under {DATA_DIR}/")
print(ALL_FILES)

52 files found under ../../data/
['../../data\\CAGE-train\\CAGE-train\\X1_train_info.tsv', '../../data\\CAGE-train\\CAGE-train\\X1_train_y.tsv', '../../data\\CAGE-train\\CAGE-train\\X1_val_info.tsv', '../../data\\CAGE-train\\CAGE-train\\X1_val_y.tsv', '../../data\\CAGE-train\\CAGE-train\\X2_train_info.tsv', '../../data\\CAGE-train\\CAGE-train\\X2_train_y.tsv', '../../data\\CAGE-train\\CAGE-train\\X2_val_info.tsv', '../../data\\CAGE-train\\CAGE-train\\X2_val_y.tsv', '../../data\\CAGE-train\\CAGE-train\\X3_test_info.tsv', '../../data\\DNase-bed\\X1.bed', '../../data\\DNase-bed\\X2.bed', '../../data\\DNase-bed\\X3.bed', '../../data\\DNase-bigwig\\X1.bw', '../../data\\DNase-bigwig\\X2.bw', '../../data\\DNase-bigwig\\X3.bigwig', '../../data\\H3K27ac-bed\\X1.bed', '../../data\\H3K27ac-bed\\X2.bed', '../../data\\H3K27ac-bed\\X3.bed', '../../data\\H3K27ac-bigwig\\X1.bigwig', '../../data\\H3K27ac-bigwig\\X2.bw', '../../data\\H3K27ac-bigwig\\X3.bw', '../../data\\H3K27me3-bed\\X1.bed', '../../dat

We then define the following two functions:
find_file only returns the file path of files falling in the CAGE-train folder, while the find_track function returns the path of tracks (DNAse, H3K4me1, H3K4me3, H3K49me3, H3K27ac, H3K27me3, H3K36me3), based on the name of the track, cell line and .bed or .bigwig extension.

In [12]:
def find_file(name):
    # Path of the unique file whose name is `name` (e.g. the CAGE-train *_info.tsv / *_y.tsv files).
    hits = [p for p in ALL_FILES if os.path.basename(p) == name]
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected exactly 1 file named {name}, found {len(hits)}: {hits}")
    return hits[0]


def find_track(mark, cell_line, kind):
    # Path of a chromatin track: data/<mark>-<kind>/<cell_line>.<ext>
    # kind: 'bed' (narrowPeak peaks) or 'bigwig' (continuous signal; extension is .bw or .bigwig).
    exts = (".bed", ".bed.gz", ".narrowpeak", ".narrowpeak.gz") if kind == "bed" else (".bw", ".bigwig")
    folder = os.path.join(DATA_DIR, f"{mark}-{kind}")
    hits = [p for p in glob.glob(os.path.join(folder, f"{cell_line}.*")) if p.lower().endswith(exts)]
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected exactly 1 {kind} file for {mark}/{cell_line}, found {len(hits)}: {hits}")
    return hits[0]

After defining functions for finding the relevant file paths, the function below asks the suer for a cell_line (X1,X2, or X3) and a split (train,val, or test). Then based on the the cell line and split, it returns a single csv file containing gene information, but also gene expression, merged on gene name.

In [13]:
def load_split(cell_line, split):
    info = pd.read_csv(find_file(f"{cell_line}_{split}_info.tsv"), sep="\t")
    if split == "test":
        return info
    y = pd.read_csv(find_file(f"{cell_line}_{split}_y.tsv"), sep="\t")
    return info.merge(y, on="gene_name", how="inner", validate="one_to_one")

data = {
    ("X1", "train"): load_split("X1", "train"),
    ("X1", "val"):   load_split("X1", "val"),
    ("X2", "train"): load_split("X2", "train"),
    ("X2", "val"):   load_split("X2", "val"),
    ("X3", "test"):  load_split("X3", "test"),
}

for key, df in data.items():
    print(key, df.shape)
data[("X1", "train")].head()

('X1', 'train') (14310, 8)
('X1', 'val') (1974, 8)
('X2', 'train') (14310, 8)
('X2', 'val') (1974, 8)
('X3', 'test') (1984, 7)


,gene_name,chr,gene_start,gene_end,TSS_start,TSS_end,strand,gex
0,SLC20A1,chr2,112645939,112663825,112658362,112658412,+,0.000000
1,C11orf58,chr11,16613132,16758340,16738643,16738693,+,2239.103328
2,ZSCAN9,chr6,28224886,28233487,28225263,28225313,+,19.798064
3,CD19,chr16,28931965,28939342,28931956,28932006,+,411.530623
4,TMEM123,chr11,102396332,102470384,102452789,102452839,-,34.214129


The following function then loads the .bed (narrowpeak) files. As the file does not contain column names, those were retrieved from the internet from the relevant documentation (https://macs3-project.github.io/MACS/docs/narrowPeak.html).

In [14]:
NARROWPEAK_COLS = ["chrom", "chromStart", "chromEnd", "name", "score", "strand",
                   "signalValue", "pValue", "qValue", "peak"]

def load_peaks(mark, cell_line):
    return pd.read_csv(find_track(mark, cell_line, "bed"), sep="\t", header=None,
                       names=NARROWPEAK_COLS, usecols=range(10), comment="#")

peaks = load_peaks("DNase", "X1")
print(peaks.shape)
peaks.head()

(56452, 10)


,chrom,chromStart,chromEnd,name,score,strand,signalValue,pValue,qValue,peak
0,chr1,181405,181555,.,0,.,12.379543,-1,-1,75
1,chr1,267990,268140,.,0,.,12.379543,-1,-1,75
2,chr1,629160,629310,.,0,.,22.263896,-1,-1,75
3,chr1,629494,629644,.,0,.,10.732151,-1,-1,75
4,chr1,629870,630020,.,0,.,3603.694491,-1,-1,75


In [19]:
bw = pybigtools.open(find_track("H3K27me3", "X1", "bigwig"))
print(bw.chroms())   # dict: chromosome -> length
print(bw.values("chr2", 1_000_000, 1_002_000, bins=1, summary="mean")) # mean DNase signal in that region, retrieved as a single value (1 bin)

for start, end, value in bw.records("chr2", 1_000_000, 1_002_000):
    print(start, end, value)

{'chr1': 248956422, 'chr10': 133797422, 'chr11': 135086622, 'chr12': 133275309, 'chr13': 114364328, 'chr14': 107043718, 'chr14_GL000009v2_random': 201709, 'chr14_GL000194v1_random': 191469, 'chr14_GL000225v1_random': 211173, 'chr14_KI270722v1_random': 194050, 'chr14_KI270723v1_random': 38115, 'chr14_KI270724v1_random': 39555, 'chr14_KI270725v1_random': 172810, 'chr15': 101991189, 'chr15_KI270727v1_random': 448248, 'chr16': 90338345, 'chr16_KI270728v1_random': 1872759, 'chr17': 83257441, 'chr17_GL000205v2_random': 185591, 'chr17_KI270729v1_random': 280839, 'chr17_KI270730v1_random': 112551, 'chr18': 80373285, 'chr19': 58617616, 'chr1_KI270706v1_random': 175055, 'chr1_KI270707v1_random': 32032, 'chr1_KI270708v1_random': 127682, 'chr1_KI270709v1_random': 66860, 'chr1_KI270711v1_random': 42210, 'chr1_KI270712v1_random': 176043, 'chr1_KI270713v1_random': 40745, 'chr1_KI270714v1_random': 41717, 'chr2': 242193529, 'chr20': 64444167, 'chr21': 46709983, 'chr22': 50818468, 'chr22_KI270731v1_rand

C:\Users\deaka\AppData\Local\Temp\ipykernel_27688\2754630515.py:3: DeprecationWarning: The default behavior of `values()` has changed: empty bins / uncovered positions are now returned as NaN instead of filled with 0. Pass `fillna=0` to keep the previous behavior, or `fillna=None` to silence this warning. See also `uncovered` to include uncovered bases in summary statistic calculations.
  print(bw.values("chr2", 1_000_000, 1_002_000, bins=1, summary="mean")) # mean DNase signal in that region, retrieved as a single value (1 bin)


## Linear baseline: mean signal around the TSS

For every gene and cell line, take the mean signal of each of the 7 assays in a window around the TSS. Log-transform it and standardize it per cell line. Fit a linear regression to `log1p(gex)` on the X1 and X2 train genes, then score the rankings with Spearman on the validation chromosomes. Fitting on one cell line and validating on the other is closer to the real task, where X3 is a cell line the model has never seen.

In [21]:
ASSAYS = ["DNase", "H3K4me3", "H3K4me1", "H3K27ac", "H3K36me3", "H3K27me3", "H3K9me3"]
WINDOW = 1000                        # bp on each side of the TSS
CACHE = f"features_tss{WINDOW}.csv"  # reading the bigWigs takes a while, so keep the result

def tss_features(genes, cell_line):
    # Mean bigWig signal in [TSS - WINDOW, TSS + WINDOW] for each assay. One row per gene.
    tss = ((genes["TSS_start"] + genes["TSS_end"]) // 2).to_numpy() # finds the TSS position for each gene
    feats = {}
    for assay in ASSAYS:
        bw = pybigtools.open(find_track(assay, cell_line, "bigwig"))
        sizes = bw.chroms()
        feats[assay] = np.array([
            bw.values(chrom, max(int(pos) - WINDOW, 0), min(int(pos) + WINDOW, sizes[chrom]),
                      bins=1, summary="mean", fillna=0)[0]
            for chrom, pos in zip(genes["chr"], tss)
        ])
    return pd.DataFrame(feats, index=genes.index)

if os.path.exists(CACHE):
    F = pd.read_csv(CACHE)
else:
    parts = []
    for (cell, split), df in data.items():
        f = tss_features(df, cell)
        f.insert(0, "gene_name", df["gene_name"].to_numpy())
        f.insert(0, "split", split)
        f.insert(0, "cell", cell)
        parts.append(f)
    F = pd.concat(parts, ignore_index=True)
    F.to_csv(CACHE, index=False)

print(F.groupby(["cell", "split"]).size())

cell  split
X1    train    14310
      val       1974
X2    train    14310
      val       1974
X3    test      1984
dtype: int64


In [23]:
def spearman(a, b):
    return pd.Series(np.asarray(a)).rank().corr(pd.Series(np.asarray(b)).rank())

# log-transform the signal, then z-score each assay within its own cell line
# (no labels involved, so the same step applies to X3)
Z = F.copy()
Z[ASSAYS] = np.log1p(Z[ASSAYS].clip(lower=0))
Z[ASSAYS] = Z.groupby("cell")[ASSAYS].transform(lambda c: (c - c.mean()) / c.std())

def design(cell, split):
    # Design matrix (intercept + 7 assays), target log1p(gex) (None for test), and the gene table.
    feats = Z[(Z["cell"] == cell) & (Z["split"] == split)][["gene_name"] + ASSAYS]
    d = data[(cell, split)].merge(feats, on="gene_name", validate="one_to_one")
    X = np.column_stack([np.ones(len(d)), d[ASSAYS].to_numpy()])
    y = np.log1p(d["gex"].to_numpy()) if "gex" in d else None
    return X, y, d

def fit(keys):
    X = np.vstack([design(*k)[0] for k in keys])
    y = np.concatenate([design(*k)[1] for k in keys])
    return np.linalg.lstsq(X, y, rcond=None)[0]

def score(w, key):
    X, y, _ = design(*key)
    return spearman(X @ w, y)

w_both = fit([("X1", "train"), ("X2", "train")])
print("Fit on X1 + X2 train, Spearman on validation:")
for cell in ("X1", "X2"):
    print(f"  {cell} val: {score(w_both, (cell, 'val')):.3f}")

print("Fit on one cell line, validate on the other:")
for tr, te in (("X1", "X2"), ("X2", "X1")):
    print(f"  train {tr} -> val {te}: {score(fit([(tr, 'train')]), (te, 'val')):.3f}")

Fit on X1 + X2 train, Spearman on validation:
  X1 val: 0.739
  X2 val: 0.701
Fit on one cell line, validate on the other:
  train X1 -> val X2: 0.682
  train X2 -> val X1: 0.738


In [18]:
# Coefficients are per standardized assay, so their sizes can be compared directly
print(pd.Series(w_both[1:], index=ASSAYS, name="coefficient").round(3).to_string())

# Predictions for the X3 test genes, in the same order as X3_test_info.tsv
test_genes = data[("X3", "test")]
X3, _, d3 = design("X3", "test")
pred = pd.Series(X3 @ w_both, index=d3["gene_name"]).reindex(test_genes["gene_name"]).to_numpy()
print(len(pred), "predictions, NaN:", int(np.isnan(pred).sum()))

DNase       0.589
H3K4me3     0.377
H3K4me1    -0.237
H3K27ac     0.524
H3K36me3    0.019
H3K27me3   -0.111
H3K9me3    -0.023
1984 predictions, NaN: 0
